<a href="https://colab.research.google.com/github/KalinaMarkova/deep_learning_course_project/blob/main/05_Model_Training_Experiment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Grained Analysis of Propaganda in News Articles
## Notebook 05: Multi-Task Learning (Joint Training) - Experiment 3

In this notebook, we will try to avoid the cascading errors from Expirement 3 and train a single unified model with two separate heads that share the same base transformer.

Head 1 will predict the binary span boundaries, while Head 2 will simultaneously predicts the 14-class technique for those tokens.

The advatage of this approach is that the model will learn that the features useful for finding propaganda are directly correlated to classifying it and context will be shared.

In [1]:
import os
from google.colab import drive
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import nltk
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm
import torch.nn as nn
import torch.optim as optim
from IPython.display import display, clear_output
from sklearn.metrics import precision_recall_fscore_support


Let's load the data, split it into 80% Training, 10% Validation, and 10% Test sets. We will use sentence-level tokenization and alignment function to generate two separate, perfectly synchronized sets of target labels.

In [2]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Extract the Raw Archive
drive_path = "/content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/datasets-v2.tgz"
print(f"Extracting raw dataset from: {drive_path}")
!tar -xzf "{drive_path}" -C /content/

# 3. Load Raw Text Articles
train_arts_dir = "/content/datasets/train-articles"
raw_articles = {}
for filepath in glob.glob(f"{train_arts_dir}/*.txt"):
    art_id = os.path.basename(filepath).replace('.txt', '').replace('article', '')
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_articles[art_id] = f.read()
print(f"Loaded {len(raw_articles)} raw training articles.")

# 4. Parse the Master Label File
train_labels_file = "/content/datasets/train-task2-TC.labels"
records = []
with open(train_labels_file, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 4:
            records.append({
                'article_id': str(parts[0]),
                'technique': parts[1],
                'start': int(parts[2]),
                'end': int(parts[3])
            })

df = pd.DataFrame(records)
print(f"Built main dataframe with {len(df)} propaganda annotations.")

# 5. Create the 80/10/10 Split
unique_articles = df['article_id'].unique()
train_ids, temp_ids = train_test_split(unique_articles, test_size=0.2, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_df = df[df['article_id'].isin(train_ids)].copy()
val_df = df[df['article_id'].isin(val_ids)].copy()
test_df = df[df['article_id'].isin(test_ids)].copy()
print(f"Split data. Train: {len(train_df)} spans | Val: {len(val_df)} spans | Test: {len(test_df)} spans.")

# 6. Initialize Tokenizer & Maps
model_checkpoint = "microsoft/deberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

unique_techniques = sorted(df['technique'].unique().tolist())
tech2id = {tech: i+1 for i, tech in enumerate(unique_techniques)}
tech2id["O"] = 0
id2tech = {id: tech for tech, id in tech2id.items()}
print(f"Tokenizer and Maps ready. Found {len(tech2id) - 1} techniques.")

# 7. Prepare NLTK and Dataset Functions
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
sentence_tokenizer = nltk.tokenize.punkt.PunktSentenceTokenizer()

def align_tokens_and_multitask_labels(raw_text, annotations, tokenizer, tech2id, max_length=512):
    char_span_labels = ["O"] * len(raw_text)
    char_tech_labels = ["O"] * len(raw_text)

    for ann in annotations:
        start, end = ann['start'], ann['end']
        technique = ann['technique']

        if start < 0 or end > len(raw_text): continue

        char_span_labels[start] = "B-Propaganda"
        for i in range(start + 1, end): char_span_labels[i] = "I-Propaganda"
        for i in range(start, end): char_tech_labels[i] = technique

    encoded = tokenizer(raw_text, return_offsets_mapping=True, truncation=True, max_length=max_length)
    span2id = {"O": 0, "B-Propaganda": 1, "I-Propaganda": 2}

    span_labels, tech_labels = [], []
    for start_char, end_char in encoded['offset_mapping']:
        if start_char == end_char:
            span_labels.append(-100)
            tech_labels.append(-100)
        else:
            span_labels.append(span2id[char_span_labels[start_char]])
            tech_labels.append(tech2id.get(char_tech_labels[start_char], 0))

    return {
        "input_ids": encoded['input_ids'],
        "attention_mask": encoded['attention_mask'],
        "span_labels": span_labels,
        "tech_labels": tech_labels
    }

class PropagandaMTLDataset(Dataset):
    def __init__(self, data_list):
        self.data_list = data_list
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        item = self.data_list[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(item["attention_mask"], dtype=torch.long),
            "span_labels": torch.tensor(item["span_labels"], dtype=torch.long),
            "tech_labels": torch.tensor(item["tech_labels"], dtype=torch.long)
        }

def prepare_mtl_dataset(df, articles_dict, tokenizer, tech2id):
    processed_data = []
    for art_id, group in tqdm(df.groupby('article_id'), desc="Processing Articles"):
        str_art_id = str(art_id)
        if str_art_id not in articles_dict: continue

        raw_text = articles_dict[str_art_id]
        article_annotations = group.to_dict('records')

        for sent_start, sent_end in sentence_tokenizer.span_tokenize(raw_text):
            sentence_text = raw_text[sent_start:sent_end]
            if not sentence_text.strip(): continue

            sentence_anns = []
            for ann in article_annotations:
                if max(sent_start, ann['start']) < min(sent_end, ann['end']):
                    sentence_anns.append({
                        'start': max(0, ann['start'] - sent_start),
                        'end': min(len(sentence_text), ann['end'] - sent_start),
                        'technique': ann['technique']
                    })

            processed_data.append(align_tokens_and_multitask_labels(
                sentence_text, sentence_anns, tokenizer, tech2id, max_length=512
            ))
    return PropagandaMTLDataset(processed_data)

def mtl_collate_fn(batch):
    return {
        "input_ids": pad_sequence([item['input_ids'] for item in batch], batch_first=True, padding_value=tokenizer.pad_token_id),
        "attention_mask": pad_sequence([item['attention_mask'] for item in batch], batch_first=True, padding_value=0),
        "span_labels": pad_sequence([item['span_labels'] for item in batch], batch_first=True, padding_value=-100),
        "tech_labels": pad_sequence([item['tech_labels'] for item in batch], batch_first=True, padding_value=-100)
    }

# 8. Build Datasets and DataLoaders
print("Preparing Datasets.")
train_dataset = prepare_mtl_dataset(train_df, raw_articles, tokenizer, tech2id)
val_dataset = prepare_mtl_dataset(val_df, raw_articles, tokenizer, tech2id)
test_dataset = prepare_mtl_dataset(test_df, raw_articles, tokenizer, tech2id)

batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=mtl_collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=mtl_collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=mtl_collate_fn)

print(f"DataLoaders ready. Training batches: {len(train_dataloader)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracting raw dataset from: /content/drive/MyDrive/Colab Notebooks/Fine_Grained_Propaganda_Analysis/datasets-v2.tgz
Loaded 371 raw training articles.
Built main dataframe with 6129 propaganda annotations.
Split data. Train: 4820 spans | Val: 662 spans | Test: 647 spans.


Tokenizer and Maps ready. Found 14 techniques.
Preparing Datasets.


Processing Articles:   0%|          | 0/285 [00:00<?, ?it/s]

Processing Articles:   0%|          | 0/36 [00:00<?, ?it/s]

Processing Articles:   0%|          | 0/36 [00:00<?, ?it/s]

DataLoaders ready. Training batches: 730


Now we need to define the architecture of the Multi-Task Learning neural network. As the DataLoaders are now outputting two sets of labels, we need to build a custom PyTorch class that can ingest the text, process it through a single shared language model, and then output two separate predictions for every token. We will initialize the pre-trained transformer base and attach two parallel linear layers (the "heads") on top of it.

In [3]:
class PropagandaMTLModel(nn.Module):
    def __init__(self, model_checkpoint, num_span_labels=3, num_tech_labels=15):
        super(PropagandaMTLModel, self).__init__()

        # 1. The Shared Base Transformer
        self.base_model = AutoModel.from_pretrained(model_checkpoint)
        hidden_size = self.base_model.config.hidden_size

        # 2. Dropout to prevent overfitting
        self.dropout = nn.Dropout(0.1)

        # 3. Head 1: Span Detection (BIO tags - 3 classes)
        self.span_classifier = nn.Linear(hidden_size, num_span_labels)

        # 4. Head 2: Technique Classification (14 techniques + 1 None = 15 classes)
        self.tech_classifier = nn.Linear(hidden_size, num_tech_labels)

    def forward(self, input_ids, attention_mask):
        # Pass inputs through the shared base model
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Extract the contextual embeddings for every token
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)

        # Pass the exact same embeddings into both heads simultaneously
        span_logits = self.span_classifier(sequence_output)
        tech_logits = self.tech_classifier(sequence_output)

        return span_logits, tech_logits

# Initialize the model and move it to the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = PropagandaMTLModel(
    model_checkpoint=model_checkpoint,
    num_span_labels=3,
    num_tech_labels=len(tech2id)
)
model.to(device)
print("Model initialized and moved to device.")

Using device: cuda


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] DebertaModel LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model initialized and moved to device.


We will use the standard AdamW optimizer and two separate CrossEntropyLoss functions.

In [4]:
# 1. The Optimizer
# 2e-5 is the standard learning rate for fine-tuning DeBERTa/RoBERTa
optimizer = optim.AdamW(model.parameters(), lr=2e-5)

# 2. The Loss Functions
# ignore_index=-100 prevents padding tokens from affecting the loss
span_loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100)
tech_loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100)

# 3. The Learning Rate Scheduler
# This warms up the learning rate slowly, then decays it, preventing catastrophic forgetting
epochs = 4
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)



Now let's start the training.

In [5]:
training_stats = []

for epoch in range(epochs):

    #               Training

    model.train()
    total_train_loss = 0

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1} - Training"):
        b_input_ids = batch['input_ids'].to(device)
        b_masks = batch['attention_mask'].to(device)
        b_span_labels = batch['span_labels'].to(device)
        b_tech_labels = batch['tech_labels'].to(device)

        model.zero_grad()

        span_logits, tech_logits = model(b_input_ids, b_masks)

        loss_span = span_loss_fn(span_logits.view(-1, 3), b_span_labels.view(-1))
        loss_tech = tech_loss_fn(tech_logits.view(-1, len(tech2id)), b_tech_labels.view(-1))

        joint_loss = loss_span + loss_tech
        total_train_loss += joint_loss.item()

        joint_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_dataloader)


    #               Validation

    model.eval()
    total_val_loss = 0
    all_span_preds, all_span_true = [], []
    all_tech_preds, all_tech_true = [], []

    with torch.no_grad():
        for batch in tqdm(val_dataloader, desc=f"Epoch {epoch + 1} - Validation"):
            b_input_ids = batch['input_ids'].to(device)
            b_masks = batch['attention_mask'].to(device)
            b_span_labels = batch['span_labels'].to(device)
            b_tech_labels = batch['tech_labels'].to(device)

            span_logits, tech_logits = model(b_input_ids, b_masks)

            # Calculate validation loss
            loss_span = span_loss_fn(span_logits.view(-1, 3), b_span_labels.view(-1))
            loss_tech = tech_loss_fn(tech_logits.view(-1, len(tech2id)), b_tech_labels.view(-1))
            total_val_loss += (loss_span + loss_tech).item()

            # Get predictions
            span_preds = torch.argmax(span_logits, dim=2).cpu().numpy()
            tech_preds = torch.argmax(tech_logits, dim=2).cpu().numpy()
            b_span_labels = b_span_labels.cpu().numpy()
            b_tech_labels = b_tech_labels.cpu().numpy()

            # Filter out -100 padding tokens
            for i in range(len(b_span_labels)):
                for j in range(len(b_span_labels[i])):
                    if b_span_labels[i][j] != -100:
                        all_span_preds.append(span_preds[i][j])
                        all_span_true.append(b_span_labels[i][j])
                        all_tech_preds.append(tech_preds[i][j])
                        all_tech_true.append(b_tech_labels[i][j])

    avg_val_loss = total_val_loss / len(val_dataloader)

    # Calculate Macro F1 Scores (Macro treats all classes equally, good for imbalanced data)
    _, _, span_f1, _ = precision_recall_fscore_support(all_span_true, all_span_preds, average='macro', zero_division=0)
    _, _, tech_f1, _ = precision_recall_fscore_support(all_tech_true, all_tech_preds, average='macro', zero_division=0)

    # Record statistics
    training_stats.append(
        {
            'Epoch': epoch + 1,
            'Train Loss': round(avg_train_loss, 4),
            'Val Loss': round(avg_val_loss, 4),
            'Span F1': round(span_f1, 4),
            'Tech F1': round(tech_f1, 4)
        }
    )

    # Clear the cell output and display the styled Pandas DataFrame
    clear_output(wait=True)
    df_stats = pd.DataFrame(data=training_stats).set_index('Epoch')
    display(df_stats)

print("Training Complete.")

,Train Loss,Val Loss,Span F1,Tech F1
Epoch,,,,
1,1.0963,0.8339,0.4260,0.0833
2,0.7023,0.8744,0.5675,0.1644
3,0.4893,1.0049,0.5776,0.2035
4,0.3356,1.1108,0.5901,0.2119


Training Complete.


The Training Loss is dropping nicely from 1.0963 down to 0.3356, meaning the model is successfully finding patterns in the training data and adjusting its weights. At the same time, F1 scores are increasing: Span F1 increased from 42% to 59%, and the Tech F1 climbed from 8% to 21%.

However, the Val Loss starts at 0.8339 and steadily climbs up to 1.1108 by epoch 4.

When Train Loss goes down but Validation Loss goes up, it is a classic sign of overfitting. The model is starting to memorize the exact wording of the training articles rather than learning general rules about propaganda. Interestingly, your F1 scores are still creeping up slightly, which means its actual predictions are getting better, but the model is getting mathematically "less confident" when it makes mistakes on the validation set.

For a 4-epoch fine-tuning run, this amount of overfitting is perfectly normal and acceptable for a baseline.

In [7]:
# 1. Model Evaluation Mode
model.eval()

pred_spans = []
unique_val_articles = val_df['article_id'].unique()

# 2. Generate Predictions Directly In-Memory
for art_id in tqdm(unique_val_articles, desc="Processing Validation Articles"):
    str_art_id = str(art_id)
    if str_art_id not in raw_articles:
        continue

    raw_text = raw_articles[str_art_id]

    for sent_start, sent_end in sentence_tokenizer.span_tokenize(raw_text):
        sentence_text = raw_text[sent_start:sent_end]
        if not sentence_text.strip():
            continue

        encoded = tokenizer(
            sentence_text,
            return_offsets_mapping=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        offsets = encoded['offset_mapping'][0].cpu().numpy()

        with torch.no_grad():
            span_logits, tech_logits = model(input_ids, attention_mask)

        span_preds = torch.argmax(span_logits, dim=2)[0].cpu().numpy()
        tech_preds = torch.argmax(tech_logits, dim=2)[0].cpu().numpy()

        current_span = None

        for idx, (offset_start, offset_end) in enumerate(offsets):
            if offset_start == offset_end:
                continue

            tech_idx = tech_preds[idx]
            span_tag = span_preds[idx]

            if tech_idx != 0 and span_tag in [1, 2]:
                tech_name = id2tech[tech_idx]
                global_start = sent_start + offset_start
                global_end = sent_start + offset_end

                if current_span is None or current_span['technique'] != tech_name or span_tag == 1:
                    if current_span is not None:
                        pred_spans.append(current_span)
                    current_span = {
                        'article_id': str_art_id,
                        'technique': tech_name,
                        'start': global_start,
                        'end': global_end
                    }
                else:
                    current_span['end'] = global_end
            else:
                if current_span is not None:
                    pred_spans.append(current_span)
                    current_span = None

        if current_span is not None:
            pred_spans.append(current_span)

# 3. Extract Ground Truth Spans
gold_spans = val_df.to_dict('records')

# Group by article ID
gold_by_art = {}
for g in gold_spans:
    gold_by_art.setdefault(str(g['article_id']), []).append(g)

pred_by_art = {}
for p in pred_spans:
    pred_by_art.setdefault(str(p['article_id']), []).append(p)

total_partial_precision = 0.0
total_partial_recall = 0.0

total_true_spans = len(gold_spans)
total_pred_spans = len(pred_spans)

# 4. Compute Character-Level Partial Overlap
for art_id, p_list in pred_by_art.items():
    g_list = gold_by_art.get(art_id, [])
    for p in p_list:
        max_overlap = 0.0
        p_len = p['end'] - p['start']
        if p_len <= 0:
            continue
        for g in g_list:
            if p['technique'] == g['technique']:
                overlap = max(0, min(p['end'], g['end']) - max(p['start'], g['start']))
                if overlap > max_overlap:
                    max_overlap = overlap
        total_partial_precision += (max_overlap / p_len)

for art_id, g_list in gold_by_art.items():
    p_list = pred_by_art.get(art_id, [])
    for g in g_list:
        max_overlap = 0.0
        g_len = g['end'] - g['start']
        if g_len <= 0:
            continue
        for p in p_list:
            if p['technique'] == g['technique']:
                overlap = max(0, min(p['end'], g['end']) - max(p['start'], g['start']))
                if overlap > max_overlap:
                    max_overlap = overlap
        total_partial_recall += (max_overlap / g_len)

# 5. Final Metric Calculations
p = total_partial_precision / total_pred_spans if total_pred_spans > 0 else 0.0
r = total_partial_recall / total_true_spans if total_true_spans > 0 else 0.0
f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

# 6. Summary Output
print("\n" + "=" * 60)
print("MULTI-TASK MODEL: SEMEVAL PARTIAL-OVERLAP SCORE")
print("=" * 60)
print(f"Total Gold Spans      : {total_true_spans}")
print(f"Total Predicted Spans : {total_pred_spans}")
print(f"Partial Precision     : {p:.4f} ({p*100:.2f}%)")
print(f"Partial Recall        : {r:.4f} ({r*100:.2f}%)")
print(f"Partial F1 Score      : {f1:.4f} ({f1*100:.2f}%)")
print("=" * 60)

Processing Validation Articles:   0%|          | 0/36 [00:00<?, ?it/s]


MULTI-TASK MODEL: SEMEVAL PARTIAL-OVERLAP SCORE
Total Gold Spans      : 662
Total Predicted Spans : 836
Partial Precision     : 0.2886 (28.86%)
Partial Recall        : 0.2082 (20.82%)
Partial F1 Score      : 0.2419 (24.19%)
